# 12 - Gobernanza: permisos por rol, linaje y contrato de oro

**Objetivo.** Requisito 5: (1) esquemas con permisos por rol, (2) linaje capturado por Unity Catalog y (3) documentación del contrato de datos de oro aplicada como metadatos.

**Roles (grupos de cuenta).** Los nombres se parametrizan porque el workspace institucional define sus propios grupos:

| Rol | Grupo por defecto | Acceso |
|---|---|---|
| Ingeniería de datos | `oceanwatch_ingenieros` | Dueño operativo: todos los privilegios en todas las capas y Volumes |
| Ciencia de datos | `oceanwatch_cientificos` | Lectura de plata, oro y referencias; escritura en `ml` (tablas y modelos) |
| Analítica / negocio | `oceanwatch_analistas` | Solo lectura de oro y referencias |
| Tablero | `oceanwatch_tablero` | Solo lectura de oro (identidad del dashboard publicado) |

Bronce, `landing` y `ops` quedan restringidos a ingeniería (principio de mínimo privilegio). Unity Catalog es aditivo (no hay DENY): por eso los permisos se otorgan por esquema y nunca a nivel de catálogo completo.

In [0]:
import os
import sys

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "src")))

from oceanwatch.contract import gold_tables, load_contract
from oceanwatch.lakehouse import get_lakehouse
from oceanwatch.notebook_params import widget

CATALOG = widget(dbutils, "catalogo", "oceanwatch_g06")
G_ING = widget(dbutils, "grupo_ingenieros", "oceanwatch_ingenieros")
G_CIE = widget(dbutils, "grupo_cientificos", "oceanwatch_cientificos")
G_ANA = widget(dbutils, "grupo_analistas", "oceanwatch_analistas")
G_TAB = widget(dbutils, "grupo_tablero", "oceanwatch_tablero")
lh = get_lakehouse(spark, catalog=CATALOG)

## 1. Permisos por rol
Cada `GRANT` se ejecuta por separado; si un grupo no existe en la cuenta se reporta sin detener el notebook.

In [0]:
C = CATALOG
GRANTS = [
    f"GRANT USE CATALOG ON CATALOG {C} TO `{g}`" for g in (G_ING, G_CIE, G_ANA, G_TAB)
] + [
    f"GRANT ALL PRIVILEGES ON SCHEMA {C}.{s} TO `{G_ING}`" for s in ("landing", "reference", "bronze", "silver", "gold", "ml", "ops")
] + [
    f"GRANT USE SCHEMA, SELECT ON SCHEMA {C}.{s} TO `{G_CIE}`" for s in ("silver", "gold", "reference")
] + [
    f"GRANT USE SCHEMA, SELECT, MODIFY, CREATE TABLE, CREATE MODEL, EXECUTE ON SCHEMA {C}.ml TO `{G_CIE}`",
] + [
    f"GRANT USE SCHEMA, SELECT ON SCHEMA {C}.{s} TO `{G_ANA}`" for s in ("gold", "reference")
] + [
    f"GRANT USE SCHEMA, SELECT ON SCHEMA {C}.gold TO `{G_TAB}`",
]

estado = []
for stmt in GRANTS:
    try:
        spark.sql(stmt)
        estado.append((stmt, "OK", ""))
    except Exception as error:  # noqa: BLE001
        estado.append((stmt, "ERROR", str(error).splitlines()[0][:200]))
display(spark.createDataFrame(estado, "sentencia string, estado string, detalle string"))

sentencia,estado,detalle
GRANT USE CATALOG ON CATALOG oceanwatch_g06 TO `oceanwatch_ingenieros`,ERROR,[RequestId=1676f359-c77d-4f59-bf7a-3eb2d1738afd ErrorClass=PRINCIPAL_DOES_NOT_EXIST.PRINCIPAL_DOES_NOT_EXIST] Could not find principal with name oceanwatch_ingenieros.
GRANT USE CATALOG ON CATALOG oceanwatch_g06 TO `oceanwatch_cientificos`,ERROR,[RequestId=ec61bdb9-5ae4-419b-8f0f-c4458372a36c ErrorClass=PRINCIPAL_DOES_NOT_EXIST.PRINCIPAL_DOES_NOT_EXIST] Could not find principal with name oceanwatch_cientificos.
GRANT USE CATALOG ON CATALOG oceanwatch_g06 TO `oceanwatch_analistas`,ERROR,[RequestId=6280b09e-64ff-4909-a8ab-03369d45dd83 ErrorClass=PRINCIPAL_DOES_NOT_EXIST.PRINCIPAL_DOES_NOT_EXIST] Could not find principal with name oceanwatch_analistas.
GRANT USE CATALOG ON CATALOG oceanwatch_g06 TO `oceanwatch_tablero`,ERROR,[RequestId=9b9cc778-d493-480b-9b87-a7b25d0308a9 ErrorClass=PRINCIPAL_DOES_NOT_EXIST.PRINCIPAL_DOES_NOT_EXIST] Could not find principal with name oceanwatch_tablero.
GRANT ALL PRIVILEGES ON SCHEMA oceanwatch_g06.landing TO `oceanwatch_ingenieros`,ERROR,[RequestId=950dc8f8-7fcc-41ed-a642-b813e5324622 ErrorClass=PRINCIPAL_DOES_NOT_EXIST.PRINCIPAL_DOES_NOT_EXIST] Could not find principal with name oceanwatch_ingenieros.
GRANT ALL PRIVILEGES ON SCHEMA oceanwatch_g06.reference TO `oceanwatch_ingenieros`,ERROR,[RequestId=925162ad-d15d-4553-a10f-cb0920ffcfe7 ErrorClass=PRINCIPAL_DOES_NOT_EXIST.PRINCIPAL_DOES_NOT_EXIST] Could not find principal with name oceanwatch_ingenieros.
GRANT ALL PRIVILEGES ON SCHEMA oceanwatch_g06.bronze TO `oceanwatch_ingenieros`,ERROR,[RequestId=838743f7-a454-4d36-9a1f-4b6b1f80aa62 ErrorClass=PRINCIPAL_DOES_NOT_EXIST.PRINCIPAL_DOES_NOT_EXIST] Could not find principal with name oceanwatch_ingenieros.
GRANT ALL PRIVILEGES ON SCHEMA oceanwatch_g06.silver TO `oceanwatch_ingenieros`,ERROR,[RequestId=e71a4d31-f2ed-43a4-b5cb-e78f4275bb8e ErrorClass=PRINCIPAL_DOES_NOT_EXIST.PRINCIPAL_DOES_NOT_EXIST] Could not find principal with name oceanwatch_ingenieros.
GRANT ALL PRIVILEGES ON SCHEMA oceanwatch_g06.gold TO `oceanwatch_ingenieros`,ERROR,[RequestId=eb2f5795-adfe-4d15-9b96-54acfad41841 ErrorClass=PRINCIPAL_DOES_NOT_EXIST.PRINCIPAL_DOES_NOT_EXIST] Could not find principal with name oceanwatch_ingenieros.
GRANT ALL PRIVILEGES ON SCHEMA oceanwatch_g06.ml TO `oceanwatch_ingenieros`,ERROR,[RequestId=3dc617c9-a91b-40d8-8154-e36b7e1222c6 ErrorClass=PRINCIPAL_DOES_NOT_EXIST.PRINCIPAL_DOES_NOT_EXIST] Could not find principal with name oceanwatch_ingenieros.


In [0]:
for s in ("bronze", "silver", "gold", "ml"):
    display(spark.sql(f"SHOW GRANTS ON SCHEMA {C}.{s}"))

Principal,ActionType,ObjectType,ObjectKey


Principal,ActionType,ObjectType,ObjectKey


Principal,ActionType,ObjectType,ObjectKey


Principal,ActionType,ObjectType,ObjectKey


## 2. Contrato de oro como metadatos (comentarios, propiedades y etiquetas)
Los comentarios de columnas ya se crean desde `contracts/gold_contract.yml` en el DDL; aquí se agregan etiquetas consultables en `information_schema`.

In [0]:
contrato = load_contract()["contrato"]
for table in gold_tables():
    fqn = lh.name("gold", table)
    spark.sql(f"""ALTER TABLE {fqn} SET TAGS ('capa' = 'oro', 'contrato' = '{contrato['id']}',
                  'contrato_version' = '{contrato['version']}', 'propietario' = '{G_ING}',
                  'clasificacion' = 'uso_interno')""")
display(spark.sql(f"""
    SELECT table_name, tag_name, tag_value FROM {C}.information_schema.table_tags
    WHERE schema_name = 'gold' ORDER BY table_name, tag_name"""))
display(spark.sql(f"""
    SELECT table_name, column_name, data_type, comment FROM {C}.information_schema.columns
    WHERE table_schema = 'gold' ORDER BY table_name, ordinal_position"""))

table_name,tag_name,tag_value
anomalias_velocidad_diarias,capa,oro
anomalias_velocidad_diarias,clasificacion,uso_interno
anomalias_velocidad_diarias,contrato,oceanwatch-gold
anomalias_velocidad_diarias,contrato_version,1.0.0
anomalias_velocidad_diarias,propietario,oceanwatch_ingenieros
estadias_puerto,capa,oro
estadias_puerto,clasificacion,uso_interno
estadias_puerto,contrato,oceanwatch-gold
estadias_puerto,contrato_version,1.0.0
estadias_puerto,propietario,oceanwatch_ingenieros


table_name,column_name,data_type,comment
anomalias_velocidad_diarias,event_date,DATE,Día UTC.
anomalias_velocidad_diarias,mmsi,STRING,MMSI del buque.
anomalias_velocidad_diarias,vessel_name,STRING,Nombre del buque.
anomalias_velocidad_diarias,vessel_type_group,STRING,Grupo de negocio.
anomalias_velocidad_diarias,n_posiciones,LONG,Posiciones del día.
anomalias_velocidad_diarias,sog_media_kn,DOUBLE,SOG medio.
anomalias_velocidad_diarias,sog_p95_kn,DOUBLE,Percentil 95 de SOG.
anomalias_velocidad_diarias,sog_max_kn,DOUBLE,SOG máximo.
anomalias_velocidad_diarias,velocidad_implicita_max_kn,DOUBLE,Máxima velocidad implícita D-04.
anomalias_velocidad_diarias,distancia_km,DOUBLE,Distancia diaria D-04.


## 3. Linaje (Unity Catalog)
Unity Catalog captura automáticamente el linaje de tablas y columnas de las escrituras del job. Se consulta en las tablas de sistema (requiere `system.access` habilitado) y se ve gráficamente en *Catalog Explorer > Lineage*.

In [0]:
try:
    display(spark.sql(f"""
        SELECT DISTINCT source_table_full_name, target_table_full_name, entity_type, max(event_time) AS ultimo_evento
        FROM system.access.table_lineage
        WHERE target_table_catalog = '{C}' OR source_table_catalog = '{C}'
        GROUP BY ALL ORDER BY target_table_full_name"""))
    display(spark.sql(f"""
        SELECT source_table_name, source_column_name, target_table_name, target_column_name
        FROM system.access.column_lineage
        WHERE target_table_catalog = '{C}' AND target_table_schema = 'gold'
        GROUP BY ALL ORDER BY target_table_name, target_column_name LIMIT 200"""))
except Exception as error:  # noqa: BLE001
    print("Tablas de sistema de linaje no disponibles para este usuario:", str(error).splitlines()[0])

source_table_full_name,target_table_full_name,entity_type,ultimo_evento
oceanwatch_g06.analytics.ais_positions_d07_delta,null,NOTEBOOK,2026-09-25T17:45:31.996Z
oceanwatch_g06.silver.ais_cuarentena,null,NOTEBOOK,2026-11-19T16:58:33.672Z
oceanwatch_g06.gold.trayectorias_diarias_buque,null,NOTEBOOK,2026-11-19T16:58:28.374Z
oceanwatch_g06.silver.ais_posiciones,null,NOTEBOOK,2026-11-19T16:58:31.431Z
oceanwatch_g06.silver.metricas_calidad,null,NOTEBOOK,2026-11-19T16:57:15.504Z
oceanwatch_g06.ops.feed_calendar,null,NOTEBOOK,2026-11-19T16:58:28.374Z
oceanwatch_g06.bronze.ais_raw,null,NOTEBOOK,2026-11-19T16:07:30.737Z
oceanwatch_g06.reference.land_cells_h3r5,null,NOTEBOOK,2026-11-19T16:07:30.737Z
null,oceanwatch_g06.analytics.ais_positions_d07_delta,NOTEBOOK,2026-09-25T17:44:07.089Z
null,oceanwatch_g06.bronze.ais_raw,NOTEBOOK,2026-11-19T16:48:26.973Z


source_table_name,source_column_name,target_table_name,target_column_name
